## INGENIERÍA DE VARIABLES — VARIANTE HORIZONTE 21 DÍAS

Misma lógica que `03_feature_engineering.ipynb`, pero cambiando el horizonte del target de 5 a 21 días (~1 mes de trading), para explorar si SVR captura más señal en un horizonte más largo (Opción 3 de mejora, Fase 6).

Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

import ta


In [2]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "features_h21"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRICES_FILE = DATA_DIR / "adjusted_prices.csv"
RETURNS_FILE = DATA_DIR / "daily_returns.csv"

TARGET_HORIZON = 21  # ÚNICO CAMBIO CONCEPTUAL respecto al notebook original (antes 5)

print("Horizonte del target:", TARGET_HORIZON, "días")


Horizonte del target: 21 días


In [3]:
prices = pd.read_csv(PRICES_FILE, index_col=0, parse_dates=True)
returns = pd.read_csv(RETURNS_FILE, index_col=0, parse_dates=True)

prices.index.name = "Date"
returns.index.name = "Date"

returns_clean = returns.dropna(how="all")
prices_clean = prices.loc[returns_clean.index]

benchmark = "^GSPC"

print("Precios:", prices_clean.shape)


Precios: (2172, 21)


## Features (idénticas al notebook original — no cambian con el horizonte)

In [4]:
HORIZONS = [5, 10, 21, 63]

return_horizon_features = {f"return_{h}d": prices_clean.pct_change(h) for h in HORIZONS}
return_horizon_features["return_1d"] = returns_clean

SMA_WINDOWS = [10, 20, 50]
sma_features, ema_features, vol_features = {}, {}, {}

for w in SMA_WINDOWS:
    sma_features[f"sma_{w}"] = prices_clean.rolling(w).mean()
    ema_features[f"ema_{w}"] = prices_clean.ewm(span=w, adjust=False).mean()
    vol_features[f"vol_{w}"] = returns_clean.rolling(w).std() * np.sqrt(252)

rsi_features = pd.DataFrame(index=prices_clean.index, columns=prices_clean.columns, dtype=float)
macd_features = pd.DataFrame(index=prices_clean.index, columns=prices_clean.columns, dtype=float)
macd_signal_features = pd.DataFrame(index=prices_clean.index, columns=prices_clean.columns, dtype=float)
bb_high_features = pd.DataFrame(index=prices_clean.index, columns=prices_clean.columns, dtype=float)
bb_low_features = pd.DataFrame(index=prices_clean.index, columns=prices_clean.columns, dtype=float)

for col in prices_clean.columns:
    series = prices_clean[col]
    rsi_features[col] = ta.momentum.RSIIndicator(close=series, window=14).rsi()
    macd = ta.trend.MACD(close=series)
    macd_features[col] = macd.macd()
    macd_signal_features[col] = macd.macd_signal()
    bb = ta.volatility.BollingerBands(close=series)
    bb_high_features[col] = bb.bollinger_hband()
    bb_low_features[col] = bb.bollinger_lband()

print("Features recalculadas (idénticas al notebook original)")


Features recalculadas (idénticas al notebook original)


## Target — único cambio real: horizonte de 21 días en vez de 5

In [5]:
target = prices_clean.pct_change(TARGET_HORIZON).shift(-TARGET_HORIZON)

target.tail()


,AAPL,ADBE,AMZN,CAT,GOOGL,HD,JNJ,JPM,KO,MA,...,MSFT,NFLX,NVDA,PEP,PG,TSLA,V,WMT,XOM,^GSPC
Date,,,,,,,,,,,,,,,,,,,,,
2026-08-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Panel largo + limpieza de NaN

In [6]:
panels = []

for ticker in prices_clean.columns:

    df_ticker = pd.DataFrame(index=prices_clean.index)
    df_ticker["ticker"] = ticker

    for name, frame in return_horizon_features.items():
        df_ticker[name] = frame[ticker]
    for name, frame in sma_features.items():
        df_ticker[name] = frame[ticker]
    for name, frame in ema_features.items():
        df_ticker[name] = frame[ticker]
    for name, frame in vol_features.items():
        df_ticker[name] = frame[ticker]

    df_ticker["rsi_14"] = rsi_features[ticker]
    df_ticker["macd"] = macd_features[ticker]
    df_ticker["macd_signal"] = macd_signal_features[ticker]
    df_ticker["bb_high"] = bb_high_features[ticker]
    df_ticker["bb_low"] = bb_low_features[ticker]

    df_ticker[f"target_return_{TARGET_HORIZON}d"] = target[ticker]

    panels.append(df_ticker)

dataset = pd.concat(panels)
dataset.index.name = "Date"
dataset = dataset.reset_index()

dataset_clean = dataset.dropna().reset_index(drop=True)

print("Dataset shape (después de limpiar NaN):", dataset_clean.shape)
dataset_clean.describe().T


Dataset shape (después de limpiar NaN): (43848, 22)


,count,mean,min,25%,50%,75%,max,std
Date,43848,2022-05-28 14:10:20.689655,2018-04-05 00:00:00,2020-04-30 18:00:00,2022-05-25 12:00:00,2024-06-25 06:00:00,2026-07-27 00:00:00,NaN
return_5d,43848.0,0.004308,-0.430883,-0.017497,0.004534,0.025194,0.564756,0.044791
return_10d,43848.0,0.008579,-0.518052,-0.022885,0.008851,0.038828,0.658833,0.062779
return_21d,43848.0,0.017846,-0.579194,-0.030214,0.016682,0.061184,1.087066,0.092349
return_63d,43848.0,0.053812,-0.614406,-0.037021,0.042537,0.124767,1.857399,0.167154
return_1d,43848.0,0.00087,-0.351166,-0.008005,0.000899,0.009835,0.243697,0.020755
sma_10,43848.0,380.809643,3.277398,87.733142,149.379818,271.547506,7545.009033,949.936064
sma_20,43848.0,379.744918,3.426798,87.639025,149.195996,270.411665,7492.848511,947.152436
sma_50,43848.0,376.61457,3.594779,87.17187,148.672483,267.973184,7472.013428,938.885175
ema_10,43848.0,380.809907,3.333195,87.819689,149.460339,271.65397,7541.122882,949.881968


## Guardar el dataset con horizonte de 21 días

In [ ]:
OUTPUT_FILE = DATA_DIR / f"features_dataset_h{TARGET_HORIZON}.csv"

dataset_clean.to_csv(OUTPUT_FILE, index=False)

print("Dataset guardado en:", OUTPUT_FILE)


Dataset guardado en: c:\Users\ofeli\Documents\GitHub\Portafolio_investigacion\data\processed\features_dataset_h21.csv


: 